# Generate irAE-arm figures

End-to-end driver that builds the IPIO irAE-arm figures from the shared survival
pipeline outputs:

1. **Labs-only paired volcano** (`paired_volcano_labs_irae.{png,pdf}` plus the
   legacy `paired_volcano_irae.{png,pdf}`) -- univariate Cox at landmarks 0d
   and +90d, colored by clinical category.
2. **Genomics-only volcano** (`volcano_genomics_irae.{png,pdf}`) -- univariate
   Cox at landmarks 0d and +90d from the genomic arm, restricted to somatic
   indicators.
3. **Lab-arm discrimination vs. baseline** (`discrimination_vs_baseline_irae.png`)
   -- test mean AUC(t) for elastic-net Cox and XGBoost, each in
   `baseline` and `baseline+labs` configurations, at both lab landmarks.
4. **Genomic-arm discrimination** (`discrimination_genomics_vs_genomics_labs_irae.png`)
   -- labs-only vs. genomics-only vs. genomics+labs multivariate models at
   genomic landmarks 0d and +90d (mean AUC(t) only).
5. **Model importance** (`model_importance_irae.{png,pdf}`) -- 2x2 grid:
   Cox log-HR coefficients (top row) and XGBoost gain (bottom row), one
   column per lab landmark.

There is no PGS-forest figure for this project (see the final markdown cell).
Each figure section is independent once the shared `Imports`, `Paths`, and
`Clinical category mapping` cells have been run.


## Imports


In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Reuse shared survival plotting utilities.
REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "survival_common").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from survival_common.plotting import RCPARAMS, parse_feature

plt.rcParams.update(RCPARAMS)


## Paths

`BASE` points at the directory containing `cox/landmark_{lm}/...` outputs.
Override for local copies.


In [ ]:
BASE = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/IPIO/"
    "survival_analysis/local_runs"
)
GENOMIC_BASE = BASE / "genomic"
GENOMIC_BASES = [GENOMIC_BASE, BASE / "genomics"]
OUT_DIR = Path("/data/gusev/USERS/jpconnor/figures/CAIA/IPIO")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS = [0, 90]
GENOMIC_ASSOCIATION_LANDMARKS = [0]  # genomics volcano/association: day-0 only
GENOMIC_MULTIVARIATE_LANDMARKS = [0, 90]  # genomics multivariate/discrimination: 0d + 90d
GENOMIC_LANDMARKS = sorted(set(GENOMIC_ASSOCIATION_LANDMARKS) | set(GENOMIC_MULTIVARIATE_LANDMARKS))  # union, for cohort loading
TOP_N = 15  # number of features shown per importance panel


## Cohort tables (per-patient)

The sections above only read Cox/XGBoost *result* CSVs. The talk figures below
(KM curves, cumulative incidence, Table 1) need the per-patient outcome table
instead: `t_irae`, `IRAE`, `event_type` (0=censored, 1=irAE, 2=death), and
baseline covariates (`pd1pdl1`, `ctla4`, `AGE_AT_TREATMENTSTART`,
`CANCER_TYPE_*`). These are written by `ipio_cohort.make_irae_outcome_df` to
`INPUTS_DIR/aggregated_landmark{D}.csv` (labs arm) and
`INPUTS_DIR/genomic/genomic_aggregated_landmark{D}.csv` (genomic arm).

`INPUTS_DIR` mirrors `IPIO_run_locally.ipynb`'s
`IPIO_PROJ_PATH / "survival_analysis" / "prediction_inputs"`. Override for
local copies, same as `BASE`.


In [ ]:
INPUTS_DIR = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/IPIO/"
    "survival_analysis/prediction_inputs"
)
GENOMIC_INPUTS_DIR = INPUTS_DIR / "genomic"

MIN_EVENTS = 10  # minimum events required per stratum arm for KM/CIF/logrank


def load_cohort(landmark, arm="labs"):
    """Per-patient outcome + feature table: t_irae, IRAE, event_type, baseline covars.

    arm="labs" -> INPUTS_DIR/aggregated_landmark{D}.csv
    arm="genomics" -> GENOMIC_INPUTS_DIR/genomic_aggregated_landmark{D}.csv, except
    landmark 0 which build_genomic_inputs.py writes as genomic_aggregated.csv
    (GENOMIC_AGGREGATED_FILENAME), not genomic_aggregated_landmark0.csv.
    """
    if arm == "labs":
        fn = INPUTS_DIR / f"aggregated_landmark{int(landmark)}.csv"
    elif int(landmark) == 0:
        fn = GENOMIC_INPUTS_DIR / "genomic_aggregated.csv"
    else:
        fn = GENOMIC_INPUTS_DIR / f"genomic_aggregated_landmark{int(landmark)}.csv"
    if not fn.exists():
        print(f"load_cohort({landmark=}, {arm=}): missing {fn}")
        return pd.DataFrame()
    df = pd.read_csv(fn)
    n_events = int(df["IRAE"].sum()) if "IRAE" in df.columns else float("nan")
    print(f"load_cohort({landmark=}, {arm=}): {len(df)} patients, {n_events} irAE events "
          f"-- {fn}")
    return df


def load_patient_risks(landmark, config="both", baseline=False):
    """Per-patient test-set risk scores from the XGBoost arm.

    File is always literally `landmark_xgboost{_baseline}_patient_risks.csv`
    (see multivariate_analysis.py); there is no per-model templating and no
    Cox-side equivalent is persisted, so this is XGBoost-only.
    """
    suffix = "_baseline" if baseline else ""
    p = BASE / "xgboost" / f"landmark_{int(landmark)}" / config / f"landmark_xgboost{suffix}_patient_risks.csv"
    if not p.exists():
        print(f"load_patient_risks({landmark=}, {config=}, {baseline=}): missing {p}")
        return pd.DataFrame()
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    print(f"load_patient_risks({landmark=}, {config=}, {baseline=}): "
          f"{len(df)} test-set rows -- {p}")
    return df


COHORT = {lm: load_cohort(lm, arm="labs") for lm in LANDMARKS}
COHORT_GENOMIC = {lm: load_cohort(lm, arm="genomics") for lm in GENOMIC_LANDMARKS}


## Table 1 -- Cohort characteristics + landmark attrition

Context slide for the talk: n patients, irAE rate, death rate, median
follow-up, drug-class breakdown, cancer-type mix at the 0d landmark, plus
landmark attrition counts from `INPUTS_DIR/landmark_attrition.json`.

`combination` therapy is **not** a surviving column in the aggregated tables
(dropped upstream in `compile_irae_data.py`); `pd1pdl1` and `ctla4` are
independent binary flags, so a patient with both set is on combination
therapy but there is no dedicated indicator column for it.


In [ ]:
import json


def drug_class_series(df):
    """Categorical drug-class label from independent pd1pdl1/ctla4 binary flags."""
    pd1 = pd.to_numeric(df.get("pd1pdl1"), errors="coerce").fillna(0).astype(int)
    ctla4 = pd.to_numeric(df.get("ctla4"), errors="coerce").fillna(0).astype(int)
    out = pd.Series("Other/unknown", index=df.index, dtype=object)
    out[(pd1 == 1) & (ctla4 == 0)] = "PD1/PDL1"
    out[(pd1 == 0) & (ctla4 == 1)] = "CTLA4"
    out[(pd1 == 1) & (ctla4 == 1)] = "Combination (PD1/PDL1 + CTLA4)"
    return out


def build_table1(df):
    rows = []
    n = len(df)
    rows.append(("N patients", n))
    if "IRAE" in df.columns:
        n_irae = int(df["IRAE"].sum())
        rows.append(("irAE events", f"{n_irae} ({100 * n_irae / n:.1f}%)" if n else n_irae))
    if "DEATH" in df.columns:
        n_death = int(df["DEATH"].sum())
        rows.append(("Deaths", f"{n_death} ({100 * n_death / n:.1f}%)" if n else n_death))
    if "t_irae" in df.columns:
        rows.append(("Median follow-up (days)", f"{df['t_irae'].median():.0f}"))
    if "AGE_AT_TREATMENTSTART" in df.columns:
        rows.append(("Median age", f"{df['AGE_AT_TREATMENTSTART'].median():.0f}"))
    if "GENDER_MALE" in df.columns:
        n_male = int(pd.to_numeric(df["GENDER_MALE"], errors="coerce").fillna(0).sum())
        rows.append(("Male", f"{n_male} ({100 * n_male / n:.1f}%)" if n else n_male))
    if {"pd1pdl1", "ctla4"}.issubset(df.columns):
        for label, count in drug_class_series(df).value_counts().items():
            rows.append((f"  Drug class: {label}", f"{count} ({100 * count / n:.1f}%)" if n else count))
    cancer_cols = [c for c in df.columns if c.startswith("CANCER_TYPE_")]
    if cancer_cols:
        cancer_flags = df[cancer_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
        n_flags_set = (cancer_flags > 0).sum(axis=1)
        n_unclassified = int((n_flags_set == 0).sum())
        n_multi = int((n_flags_set > 1).sum())
        if n_multi:
            print(f"build_table1: {n_multi} patients have >1 CANCER_TYPE_* flag set "
                  f"(each counted once per flagged type -- category counts will exceed n)")
        if n_unclassified:
            print(f"build_table1: {n_unclassified} patients have 0 CANCER_TYPE_* flags set "
                  f"(reported as 'Other/unclassified')")
        for c in cancer_cols:
            count = int(cancer_flags[c].sum())
            if count > 0:
                rows.append((f"  Cancer type: {(c[len('CANCER_TYPE_'):] if c.startswith('CANCER_TYPE_') else c)}", f"{count} ({100 * count / n:.1f}%)" if n else count))
        if n_unclassified:
            rows.append(("  Cancer type: Other/unclassified", f"{n_unclassified} ({100 * n_unclassified / n:.1f}%)" if n else n_unclassified))
    return pd.DataFrame(rows, columns=["Characteristic", "Value"])


table1_df = build_table1(COHORT[0]) if not COHORT[0].empty else pd.DataFrame()

attrition_path = INPUTS_DIR / "landmark_attrition.json"
attrition = {}
if attrition_path.exists():
    attrition = json.loads(attrition_path.read_text())
    print(f"Loaded attrition from {attrition_path}")
else:
    print(f"Table 1: missing {attrition_path}")

if not table1_df.empty:
    fig, ax = plt.subplots(figsize=(6.5, 0.42 * (len(table1_df) + len(attrition) + 2) + 1))
    ax.axis("off")
    display_rows = table1_df.values.tolist()
    if attrition:
        display_rows.append(["", ""])
        display_rows.append(["Landmark attrition", ""])
        display_rows.append(["  n loaded cohort", attrition.get("n_loaded_cohort", "")])
        for lm, count in attrition.get("eligible_by_landmark", {}).items():
            display_rows.append([f"  eligible @ landmark {lm}d", count])
        display_rows.append(["  common across landmarks", attrition.get("n_common_across_landmarks", "")])
        for split_name, count in attrition.get("split_sizes", {}).items():
            display_rows.append([f"  split: {split_name}", count])
    tbl = ax.table(cellText=display_rows, colLabels=["Characteristic", "Value"],
                    loc="center", cellLoc="left", colLoc="left")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.3)
    ax.set_title("Table 1 -- IPIO cohort characteristics (landmark 0d)", fontsize=12, weight="bold", pad=12)
    fig.tight_layout()
    for ext in ("png",):
        out = OUT_DIR / f"cohort_table1_irae.{ext}"
        fig.savefig(out, bbox_inches="tight")
        print(f"wrote {out}")
    plt.show()
else:
    print("Table 1: skipped, no landmark-0 cohort data available")


## Clinical category mapping

IPIO is a pan-cancer immunotherapy cohort, not prostate-specific, so this
taxonomy does **not** reuse COMPASS's `ANDROGEN = {"PSA", "Testosterone"}`
bucket. Instead it groups labs into CBC / CMP / LFT / Thyroid / Vitals /
Other, with a dedicated `Thyroid` bucket since thyroid dysfunction is one of
the most common irAEs. **This list is a placeholder** adapted from general
irAE-monitoring conventions, not from the actual canonical-lab list for this
cohort -- revisit once that's known.


In [ ]:
# NOTE: adapted for an immunotherapy / irAE cohort instead of COMPASS's
# prostate-specific taxonomy (no "Androgen axis" bucket here). Revisit this
# list once the real canonical-lab set for the IPIO cohort is confirmed.
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
THYROID = {"TSH", "Free T4"}  # thyroiditis is a common irAE -- own bucket
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
OTHER = {"CRP", "ESR"}  # + anything unmapped falls here by default

CATEGORY_MAP = {}
for label, members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Thyroid", THYROID), ("Vitals", VITALS), ("Other", OTHER),
]:
    for m in members:
        CATEGORY_MAP[m] = label

# "Genomic" is not populated via CATEGORY_MAP (no lab_name collides with a
# gene name) -- volcano code paths that plot genomic indicators set
# category="Genomic" directly instead of going through assign_category().
DRAW_ORDER = ["Other", "Vitals", "Thyroid", "CMP", "LFT", "CBC", "Genomic"]
LEGEND_ORDER = ["CBC", "LFT", "CMP", "Thyroid", "Vitals", "Other", "Genomic"]

CATEGORY_COLORS = {
    "CBC":     "#16a085",
    "LFT":     "#e67e22",
    "CMP":     "#7d3c98",
    "Thyroid": "#8e1c2b",
    "Vitals":  "#5d6d7e",
    "Other":   "#95a5a6",
    "Genomic": "#2471a3",
}
NS_COLOR = "#d5d8dc"


def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(lab_name, "Other")


def format_label(lab_name, feature_stat):
    if not feature_stat or pd.isna(feature_stat) or feature_stat == "":
        return lab_name
    return f"{lab_name} ({feature_stat})"


## Figure 1 -- Volcano panels (univariate Cox)

The labs-only volcano uses the full landmark cohort at 0d and +90d. The
separate genomics-only volcano uses the genomic arm at 0d and +90d and keeps
only somatic indicator features (`<GENE>_<SV|SNV|AMP|DEL>`).

Sources:
- Labs: `cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`
- Genomics: `genomic/cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`

Both are filtered to `endpoint == "irae"`.


In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 6
# Placeholder -- re-pick once real univariate hits are known for this cohort.
# LFT + thyroid labs are plausible irAE signals so used here as a stand-in.
ALWAYS_LABEL = {"TSH", "ALT", "Creatinine"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5
MIN_LABEL_SPACING_NLP = 0.55
PANEL_XLIM = (-2.0, 2.0)  # zoomed view for the talk; COEF_CAP = 4 still governs stability filtering
Y_MAX_CAP = 30  # -log10(p) ceiling; anything above this is drawn at the cap as a triangle


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    """Stack labels at the left/right panel edges with leader lines.

    Selection rule: dedupe to the most-significant stat per lab, then include
    the `always_label` whitelist + top_k by p-value.
    """
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    best = (sig.sort_values("p_value")
               .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = best.loc[~best["lab_name"].isin(always_label)].head(top_k)
    to_label = pd.concat([always_sig, extra]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        if side == "left":
            label_x = xlim[0] + 0.04 * xspan
            ha = "left"
        else:
            label_x = xlim[1] - 0.04 * xspan
            ha = "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = min(r["neglog10p"], Y_MAX_CAP)
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            # format_label collapses cleanly when feature_stat is blank, which
            # genomic-indicator rows (no per-lab feature_stat) rely on.
            text = format_label(r["lab_name"], r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text, xy=(r["coef_feature"], min(r["neglog10p"], Y_MAX_CAP)),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    left = to_label.loc[to_label["coef_feature"] < 0]
    right = to_label.loc[to_label["coef_feature"] >= 0]
    _stack(left, "left")
    _stack(right, "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.copy()
    # Allow callers to pre-populate "category" (e.g. genomic-indicator rows,
    # which aren't a lab_name assign_category() can resolve).
    if "category" not in sub.columns:
        sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05
    # Cap extreme -log10(p) so a handful of huge values don't squash the rest.
    sub["capped"] = sub["neglog10p"] > Y_MAX_CAP
    sub["y"] = sub["neglog10p"].clip(upper=Y_MAX_CAP)

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["y"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        base_kw = dict(
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
        )
        in_range = cat_sig.loc[~cat_sig["capped"]]
        if not in_range.empty:
            ax.scatter(
                in_range["coef_feature"], in_range["y"],
                s=40, marker="o", zorder=3, **base_kw,
            )
        over = cat_sig.loc[cat_sig["capped"]]
        if not over.empty:
            ax.scatter(
                over["coef_feature"], over["y"],
                s=60, marker="^", zorder=4, **base_kw,
            )
            for _, r in over.iterrows():
                ax.annotate(
                    f"p={r['p_value']:.1e}",
                    xy=(r["coef_feature"], Y_MAX_CAP),
                    xytext=(0, -8), textcoords="offset points",
                    ha="center", va="top",
                    fontsize=7.5, color=CATEGORY_COLORS[cat],
                    zorder=7,
                )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["y"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0, color="grey", linestyle="-", linewidth=0.7, zorder=0)
    for x in (-0.5, 0.5):
        ax.axvline(x, color="grey", linestyle="--", linewidth=0.6,
                   alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    bd_str = "  ".join(f"{c} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")


In [ ]:
GENOMIC_FEATURE_RE = re.compile(r"^[A-Za-z0-9]+_(SV|SNV|AMP|DEL)$")
BASELINE_COVARIATE_NAMES = {"age", "GENDER_MALE", "pd1pdl1", "ctla4"}
GENOMIC_UNIVARIATE_CONFIGS = [
    "genomics",
    "genomic",
    "genomics_only",
    "genomic_only",
    "both",
    "genomics_plus_labs",
    "genomic_plus_labs",
]


def _is_baseline_covariate(feature):
    return str(feature) in BASELINE_COVARIATE_NAMES or str(feature).startswith("CANCER_TYPE_")


def _is_genomic_feature(feature):
    return bool(GENOMIC_FEATURE_RE.match(str(feature)))


def _format_genomic_label(feature):
    gene, variant = str(feature).rsplit("_", 1)
    return f"{gene} {variant}"


def _first_existing_path(candidates, *, label):
    checked = []
    for p in candidates:
        checked.append(str(p))
        if p.exists():
            print(f"{label}: using {p}")
            return p
    preview = "\n  ".join(checked[:12])
    if len(checked) > 12:
        preview += f"\n  ... ({len(checked) - 12} more)"
    raise FileNotFoundError(f"No {label} file found. Checked:\n  {preview}")


def _load_uni_from_candidates(base_dirs, landmark, config_dirs, *, label):
    candidates = [
        base_dir / "cox" / f"landmark_{landmark}" / config_dir / "cox_agg_univariate_nobs_adjusted.csv"
        for base_dir in base_dirs
        for config_dir in config_dirs
    ]
    p = _first_existing_path(candidates, label=label)
    df = pd.read_csv(p)
    df["landmark_days"] = landmark
    df["source_path"] = str(p)
    return df


def _load_lab_uni(landmark):
    return _load_uni_from_candidates([BASE], landmark, ["both"], label=f"lab univariate landmark {landmark}")


def _load_genomic_uni(landmark):
    return _load_uni_from_candidates(
        GENOMIC_BASES,
        landmark,
        GENOMIC_UNIVARIATE_CONFIGS,
        label=f"genomic univariate landmark {landmark}",
    )


def _ensure_lab_columns(df):
    df = df.copy()
    if "lab_name" not in df.columns or "feature_stat" not in df.columns:
        parsed = df["feature"].map(parse_feature)
        df["lab_name"] = parsed.str[0]
        df["feature_stat"] = parsed.str[1]
    df["lab_name"] = df["lab_name"].fillna(df["feature"].astype(str))
    df["feature_stat"] = df["feature_stat"].fillna("")
    return df


def _prepare_volcano_rows(df, *, kind):
    df = df.loc[df["endpoint"] == "irae"].copy()
    df = df.dropna(subset=["feature", "coef_feature", "p_value", "q_value"])
    genomic_mask = df["feature"].map(_is_genomic_feature)
    baseline_mask = df["feature"].map(_is_baseline_covariate)
    if kind == "labs":
        df = _ensure_lab_columns(df.loc[~genomic_mask & ~baseline_mask])
        df["category"] = df["lab_name"].map(assign_category)
    elif kind == "genomics":
        df = df.loc[genomic_mask].copy()
        df["lab_name"] = df["feature"].map(_format_genomic_label)
        df["feature_stat"] = ""
        df["category"] = "Genomic"
    else:
        raise ValueError(f"Unknown volcano kind: {kind}")
    return df


def _filter_stable_volcano_rows(df, *, label):
    if df.empty:
        print(f"{label}: 0 rows")
        return df
    ci_ratio = df["ci_upper"] / df["ci_lower"]
    mask = (
        df["coef_feature"].abs().le(COEF_CAP)
        & ci_ratio.lt(CI_RATIO_CAP)
        & np.isfinite(ci_ratio)
    )
    print(f"{label}: dropping {int((~mask).sum())} / {len(df)} unstable rows")
    out = df.loc[mask].copy()
    print(f"{label}: {len(out)} rows remaining")
    return out


def _pick_always_label(landmark, k=3):
    """Data-driven replacement for the ALWAYS_LABEL placeholder: top-k non-genomic,
    non-baseline labs by univariate p-value at `landmark`. Falls back to the
    original placeholder set (printed, not silently) if loading fails."""
    try:
        uni = _load_lab_uni(landmark)
    except FileNotFoundError as exc:
        print(f"_pick_always_label: falling back to placeholder ALWAYS_LABEL ({exc})")
        return None
    uni = uni.loc[uni["endpoint"] == "irae"].copy()
    genomic_mask = uni["feature"].map(_is_genomic_feature)
    baseline_mask = uni["feature"].map(_is_baseline_covariate)
    uni = _ensure_lab_columns(uni.loc[~genomic_mask & ~baseline_mask])
    uni = uni.dropna(subset=["p_value"]).sort_values("p_value")
    top = uni.drop_duplicates("lab_name", keep="first").head(k)["lab_name"].tolist()
    return set(top) if top else None


# Replace the Fig-1-cell placeholder ALWAYS_LABEL with the actual top univariate
# hits at the primary landmark, so the talk labels real signal instead of a
# stand-in. Falls back to the placeholder (with a loud print) if data is
# unavailable -- verify the canonical-lab set (see Clinical category mapping
# cell) before presenting either way.
_picked_always_label = _pick_always_label(LANDMARKS[0])
if _picked_always_label:
    print(f"Replacing placeholder ALWAYS_LABEL {ALWAYS_LABEL} with data-driven top hits: {_picked_always_label}")
    ALWAYS_LABEL = _picked_always_label
else:
    print(f"Keeping placeholder ALWAYS_LABEL {ALWAYS_LABEL} -- verify against the real canonical-lab set before the talk.")


lab_uni = pd.concat([_load_lab_uni(lm) for lm in LANDMARKS], ignore_index=True)
lab_uni = _prepare_volcano_rows(lab_uni, kind="labs")
print(f"labs: {len(lab_uni)} (lab x stat) rows across landmarks "
      f"{sorted(lab_uni['landmark_days'].unique())}")

# Filter unstable Cox estimates: |log HR| > 4 or CI spans > 2 orders of magnitude.
COEF_CAP = 4.0
CI_RATIO_CAP = 100
lab_uni = _filter_stable_volcano_rows(lab_uni, label="labs")

panels = [(0, "0 days"), (90, "+90 days")]
fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
for ax, (lm, title) in zip(axes, panels):
    sub = lab_uni.loc[lab_uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no lab data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER if c != "Genomic"]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q >= 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png",):
    for stem in ("paired_volcano_labs_irae", "paired_volcano_irae"):
        out = OUT_DIR / f"{stem}.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
plt.show()


genomic_uni = pd.concat([_load_genomic_uni(lm) for lm in GENOMIC_ASSOCIATION_LANDMARKS], ignore_index=True)
genomic_uni = _prepare_volcano_rows(genomic_uni, kind="genomics")
print(f"genomics: {len(genomic_uni)} somatic-indicator rows across landmarks "
      f"{sorted(genomic_uni['landmark_days'].unique()) if not genomic_uni.empty else []}")
genomic_uni = _filter_stable_volcano_rows(genomic_uni, label="genomics")

genomic_panels = [(lm, f"Genomics · {'+' if lm > 0 else ''}{lm} days") for lm in GENOMIC_ASSOCIATION_LANDMARKS]
fig, axes = plt.subplots(
    1, len(genomic_panels), figsize=(7.5 * len(genomic_panels), 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
if len(genomic_panels) == 1:
    axes = [axes]
for ax, (lm, title) in zip(axes, genomic_panels):
    sub = genomic_uni.loc[genomic_uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no genomic univariate rows for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [
    Patch(facecolor=CATEGORY_COLORS["Genomic"], edgecolor="white", label="Genomic"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=NS_COLOR,
           markersize=8, label="ns (q >= 0.05)"),
]
fig.legend(handles=handles, loc="lower center", ncol=len(handles),
           bbox_to_anchor=(0.5, -0.04), fontsize=10)
fig.tight_layout(rect=[0, 0.04, 1, 1])

for ext in ("png",):
    out = OUT_DIR / f"volcano_genomics_irae.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Table 2 -- Lab feature counts by clinical category (univariate vs. multivariate)

How many lab features were tested (univariate Cox, one feature at a time) vs.
actually retained in the multivariate model, broken down by clinical category
(CBC/CMP/LFT/Vitals/Androgen axis/Other), per landmark.

- **Univariate**: every non-genomic, non-baseline feature present in
  `cox_agg_univariate_nobs_adjusted.csv` (one row per lab x summary-stat).
- **Multivariate (Cox)**: non-zero-coefficient features retained in
  `cox_agg_multivariable.csv` after elastic-net selection.
- **Multivariate (XGBoost)**: positive-gain features in
  `landmark_xgboost_feature_importance.csv`.

Counts are at the (lab, summary-stat) feature level, not deduplicated to
unique lab names -- this matches how the volcano/importance panels count
features.


In [ ]:
def _load_cox_multivariable(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    if not p.exists():
        print(f"lab counts: missing {p}")
        return pd.DataFrame()
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def _load_xgb_feature_importance(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_feature_importance.csv"
    if not p.exists():
        print(f"lab counts: missing {p}")
        return pd.DataFrame()
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    df = df.loc[df["gain"].fillna(0) > 0]
    return df


def _lab_only(df):
    if df.empty or "feature" not in df.columns:
        return df
    genomic_mask = df["feature"].map(_is_genomic_feature)
    baseline_mask = df["feature"].map(_is_baseline_covariate)
    return _ensure_lab_columns(df.loc[~genomic_mask & ~baseline_mask])


def build_lab_count_table(landmark):
    uni = _lab_only(_load_lab_uni(landmark))
    uni = uni.loc[uni["endpoint"] == "irae"] if "endpoint" in uni.columns else uni
    cox_mv = _lab_only(_load_cox_multivariable(landmark))
    xgb_mv = _lab_only(_load_xgb_feature_importance(landmark))

    def _counts_by_category(df):
        if df.empty:
            return pd.Series(dtype=int)
        cat = df["lab_name"].map(assign_category)
        return cat.value_counts()

    uni_counts = _counts_by_category(uni)
    cox_counts = _counts_by_category(cox_mv)
    xgb_counts = _counts_by_category(xgb_mv)

    categories = [c for c in LEGEND_ORDER if c != "Genomic"]
    rows = []
    for cat in categories:
        rows.append((
            cat,
            int(uni_counts.get(cat, 0)),
            int(cox_counts.get(cat, 0)),
            int(xgb_counts.get(cat, 0)),
        ))
    rows.append((
        "Total",
        int(uni_counts.sum()),
        int(cox_counts.sum()),
        int(xgb_counts.sum()),
    ))
    return pd.DataFrame(
        rows, columns=["Clinical category", "Univariate (tested)", "Multivariate (Cox)", "Multivariate (XGBoost)"]
    )


for lm in LANDMARKS:
    sign = "+" if lm > 0 else ""
    lab_count_df = build_lab_count_table(lm)
    if lab_count_df.empty or lab_count_df["Univariate (tested)"].sum() == 0:
        print(f"Table 2 (landmark {lm}d): skipped, no lab feature data available")
        continue

    fig, ax = plt.subplots(figsize=(7.5, 0.42 * (len(lab_count_df) + 2) + 1))
    ax.axis("off")
    tbl = ax.table(cellText=lab_count_df.values.tolist(), colLabels=lab_count_df.columns.tolist(),
                    loc="center", cellLoc="left", colLoc="left")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.3)
    ax.set_title(f"Table 2 -- Lab feature counts by clinical category -- {sign}{lm} days",
                 fontsize=12, weight="bold", pad=12)
    fig.tight_layout()
    for ext in ("png",):
        out = OUT_DIR / f"table2_lab_counts_irae_landmark{lm}.{ext}"
        fig.savefig(out, bbox_inches="tight")
        print(f"wrote {out}")
    plt.show()


## Table 3 -- Genomics+labs multivariate feature counts (by mutation type)

Companion table for the genomics-arm multivariate runs (Figure 2's genomics
comparison panel): how many genomic features were retained in the
**genomics+labs** multivariate model, broken down by mutation type
(SV/SNV/AMP/DEL), plus the cohort size (n patients, n irAEs, n deaths) for
that run.

Sources (irae endpoint, genomic landmark, `genomics_plus_labs`-style config):
- **Cox**: `genomic/cox/landmark_{lm}/{config}/cox_agg_multivariable.csv`
- **XGBoost**: `genomic/xgboost/landmark_{lm}/{config}/landmark_xgboost_feature_importance.csv`
- **Cohort**: `genomic_aggregated_landmark{lm}.csv` (via `COHORT_GENOMIC`)


In [ ]:
def _first_existing_genomic_path(base_dirs, model_dir, landmark, config_dirs, filename, *, label):
    candidates = [
        base_dir / model_dir / f"landmark_{landmark}" / config_dir / filename
        for base_dir in base_dirs
        for config_dir in config_dirs
    ]
    for p in candidates:
        if p.exists():
            print(f"{label}: using {p}")
            return p
    preview = "\n  ".join(str(p) for p in candidates[:12])
    print(f"{label}: missing; checked:\n  {preview}")
    return None


# Local copy -- GENOMIC_PLUS_LABS_CONFIGS proper is defined later (Figure 2 cell);
# duplicated here (kept in sync with that cell) so this table doesn't depend on
# a forward reference for top-to-bottom execution.
_GENOMIC_PLUS_LABS_CONFIGS_LOCAL = [
    "genomics_plus_labs",
    "genomic_plus_labs",
    "genomics_labs",
    "labs_plus_genomics",
    "both",
]


def _load_genomic_multivariate(landmark, model_dir, filename, value_col):
    p = _first_existing_genomic_path(
        GENOMIC_BASES, model_dir, landmark, _GENOMIC_PLUS_LABS_CONFIGS_LOCAL, filename,
        label=f"genomics+labs {model_dir} multivariate landmark {landmark}",
    )
    if p is None:
        return pd.DataFrame()
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    df = df.loc[df[value_col].fillna(0) != 0]
    df = df.loc[df["feature"].map(_is_genomic_feature)]
    return df


def _mutation_type(feature):
    return str(feature).rsplit("_", 1)[-1]


def build_genomic_feature_table(landmark):
    cox_mv = _load_genomic_multivariate(landmark, "cox", "cox_agg_multivariable.csv", "coef")
    xgb_mv = _load_genomic_multivariate(landmark, "xgboost", "landmark_xgboost_feature_importance.csv", "gain")

    def _counts_by_mutation(df):
        if df.empty:
            return pd.Series(dtype=int)
        return df["feature"].map(_mutation_type).value_counts()

    cox_counts = _counts_by_mutation(cox_mv)
    xgb_counts = _counts_by_mutation(xgb_mv)
    mutation_types = sorted(set(cox_counts.index) | set(xgb_counts.index))

    rows = [
        (mt, int(cox_counts.get(mt, 0)), int(xgb_counts.get(mt, 0)))
        for mt in mutation_types
    ]
    rows.append(("Total", int(cox_counts.sum()), int(xgb_counts.sum())))
    return pd.DataFrame(rows, columns=["Mutation type", "Multivariate (Cox)", "Multivariate (XGBoost)"])


for lm in GENOMIC_MULTIVARIATE_LANDMARKS:
    sign = "+" if lm > 0 else ""
    genomic_feature_df = build_genomic_feature_table(lm)

    genomic_cohort = COHORT_GENOMIC.get(lm, pd.DataFrame())
    n_patients = len(genomic_cohort)
    n_irae = int(genomic_cohort["IRAE"].sum()) if "IRAE" in genomic_cohort.columns else None
    n_death = int(genomic_cohort["DEATH"].sum()) if "DEATH" in genomic_cohort.columns else None

    if genomic_feature_df.empty and n_patients == 0:
        print(f"Table 3 (landmark {lm}d): skipped, no genomics+labs multivariate data available")
        continue

    cohort_rows = [("N patients (genomic cohort)", n_patients)]
    if n_irae is not None:
        cohort_rows.append(("irAE events", f"{n_irae} ({100 * n_irae / n_patients:.1f}%)" if n_patients else n_irae))
    if n_death is not None:
        cohort_rows.append(("Deaths", f"{n_death} ({100 * n_death / n_patients:.1f}%)" if n_patients else n_death))
    cohort_df = pd.DataFrame(cohort_rows, columns=["Characteristic", "Value"])

    n_feature_rows = len(genomic_feature_df) if not genomic_feature_df.empty else 1
    fig, axes = plt.subplots(
        1, 2, figsize=(11.5, 0.42 * max(n_feature_rows, len(cohort_rows)) + 1.6),
        gridspec_kw={"width_ratios": [1.3, 1]},
    )
    for ax in axes:
        ax.axis("off")

    if not genomic_feature_df.empty:
        tbl = axes[0].table(cellText=genomic_feature_df.values.tolist(),
                             colLabels=genomic_feature_df.columns.tolist(),
                             loc="center", cellLoc="left", colLoc="left")
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        tbl.scale(1, 1.3)
    else:
        axes[0].text(0.5, 0.5, "(no genomic multivariate features)",
                     ha="center", va="center", transform=axes[0].transAxes, color="#7f8c8d")
    axes[0].set_title("Genomic features retained", fontsize=11, weight="bold", pad=10)

    tbl2 = axes[1].table(cellText=cohort_df.values.tolist(), colLabels=cohort_df.columns.tolist(),
                          loc="center", cellLoc="left", colLoc="left")
    tbl2.auto_set_font_size(False)
    tbl2.set_fontsize(9)
    tbl2.scale(1, 1.3)
    axes[1].set_title("Genomic cohort", fontsize=11, weight="bold", pad=10)

    fig.suptitle(f"Table 3 -- Genomics+labs multivariate summary -- {sign}{lm} days",
                 fontsize=12.5, weight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    for ext in ("png",):
        out = OUT_DIR / f"table3_genomic_counts_irae_landmark{lm}.{ext}"
        fig.savefig(out, bbox_inches="tight")
        print(f"wrote {out}")
    plt.show()


## Figure 4 -- KM & cumulative-incidence curves

Stratified time-to-irAE curves, the headline addition for the talk. Placed
after Figure 1 so it can reuse that section's `_load_lab_uni`,
`_ensure_lab_columns`, `_is_genomic_feature`, and `_is_baseline_covariate`
helpers to pick top univariate lab hits.

**Competing risk note:** for time-to-irAE, death is a competing event (a
patient who dies cannot later have an irAE). A naive Kaplan-Meier that
censors death **overestimates** cumulative irAE incidence, so:

- The primary panels below are Kaplan-Meier curves labeled **"irAE-free
  probability"** (not "survival probability"), with death treated as
  censoring -- read as "the probability of remaining irAE-free," not a true
  survival curve.
- A cumulative-incidence (Aalen-Johansen) **appendix** figure treats death as
  a genuine competing event using the `event_type` column already built by
  `ipio_cohort.make_irae_outcome_df` (0=censored, 1=irAE, 2=death).

Every stratified curve drops arms with fewer than `MIN_EVENTS` irAE events
and prints what was dropped -- sparse arms produce meaningless KM tails.
**Landmark 0d is the headline; +90d is left to the appendix.**


In [ ]:
from survival_common.plotting import overlay_km

try:
    from lifelines import AalenJohansenFitter
    from lifelines.statistics import multivariate_logrank_test
    LIFELINES_OK = True
except ModuleNotFoundError as exc:
    AalenJohansenFitter = None
    multivariate_logrank_test = None
    LIFELINES_OK = False
    print(f"lifelines unavailable ({exc}); KM/CIF cells will be skipped.")


def dichotomize(df, col, cut="median"):
    """Categorical split of a continuous column at its median (or an explicit cutoff)."""
    vals = pd.to_numeric(df[col], errors="coerce")
    cutoff = vals.median() if cut == "median" else float(cut)
    out = pd.Series(pd.NA, index=df.index, dtype=object)
    out[vals.notna() & (vals >= cutoff)] = f"{col} >= {cutoff:.2g}"
    out[vals.notna() & (vals < cutoff)] = f"{col} < {cutoff:.2g}"
    return out, cutoff


def tertile_split(df, col):
    """3-way categorical split of a continuous column into tertiles (Low/Mid/High)."""
    vals = pd.to_numeric(df[col], errors="coerce")
    out = pd.Series(pd.NA, index=df.index, dtype=object)
    try:
        labels = pd.qcut(vals, 3, labels=[f"{col} (Low)", f"{col} (Mid)", f"{col} (High)"])
    except ValueError:
        return out, None
    out[vals.notna()] = labels.astype(str)[vals.notna()]
    return out, labels.cat.categories.tolist() if hasattr(labels, "cat") else None


def top_univariate_labs(landmark, k=3):
    """Top-k non-genomic, non-baseline lab features by univariate p-value at `landmark`.

    Reuses _load_lab_uni / _ensure_lab_columns / _is_genomic_feature /
    _is_baseline_covariate from the Figure 1 (volcano) section above.
    """
    uni = _load_lab_uni(landmark)
    uni = uni.loc[uni["endpoint"] == "irae"].copy()
    genomic_mask = uni["feature"].map(_is_genomic_feature)
    baseline_mask = uni["feature"].map(_is_baseline_covariate)
    uni = uni.loc[~genomic_mask & ~baseline_mask]
    uni = _ensure_lab_columns(uni)
    uni = uni.dropna(subset=["p_value"]).sort_values("p_value")
    best = uni.drop_duplicates("lab_name", keep="first").head(k)
    return best["lab_name"].tolist(), best


def _strata_min_events(df, strata_col, duration_col="t_irae", event_col="IRAE", min_events=MIN_EVENTS):
    """(kept_df, dropped_counts) after removing arms with < min_events events."""
    counts = df.groupby(strata_col)[event_col].sum()
    keep_labels = counts.loc[counts >= min_events].index
    dropped = counts.loc[counts < min_events]
    if not dropped.empty:
        print(f"  dropping strata (< {min_events} events): "
              + ", ".join(f"{lbl!r} ({int(n)} events)" for lbl, n in dropped.items()))
    kept = df.loc[df[strata_col].isin(keep_labels)].copy()
    return kept, dropped


def km_by_strata(ax, df, strata_col, *, title, duration_col="t_irae", event_col="IRAE",
                  min_events=MIN_EVENTS, colors=None):
    """Overlay KM (irAE-free probability) curves for each level of `strata_col`, with log-rank p."""
    if not LIFELINES_OK or df.empty or strata_col not in df.columns:
        ax.text(0.5, 0.5, "(no data)", ha="center", va="center",
                transform=ax.transAxes, color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    sub = df.dropna(subset=[strata_col, duration_col, event_col]).copy()
    sub = sub.loc[sub[duration_col] > 0]
    print(f"{title}:")
    kept, _dropped = _strata_min_events(sub, strata_col, duration_col, event_col, min_events)
    if kept.empty or kept[strata_col].nunique() < 1:
        ax.text(0.5, 0.5, "(insufficient events after filtering)",
                ha="center", va="center", transform=ax.transAxes, color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    survival_by_label = {
        str(level): (grp[duration_col], grp[event_col])
        for level, grp in kept.groupby(strata_col)
    }
    overlay_km(ax, survival_by_label, colors=colors, title=title, xlabel="Days",
               ylabel="irAE-free probability")

    if kept[strata_col].nunique() >= 2:
        result = multivariate_logrank_test(kept[duration_col], kept[strata_col], kept[event_col])
        ax.text(0.98, 0.02, f"log-rank p = {result.p_value:.3g}",
                transform=ax.transAxes, ha="right", va="bottom",
                fontsize=8.5, color="#5d6d7e", family="monospace")
    ax.legend(fontsize=7.5, loc="lower left")


def cif_by_strata(ax, df, strata_col, *, title, duration_col="t_irae", event_type_col="event_type",
                   event_of_interest=1, min_events=MIN_EVENTS, colors=None):
    """Overlay Aalen-Johansen cumulative-incidence curves for irAE, death as competing event."""
    if not LIFELINES_OK or df.empty or strata_col not in df.columns or event_type_col not in df.columns:
        ax.text(0.5, 0.5, "(no data)", ha="center", va="center",
                transform=ax.transAxes, color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    sub = df.dropna(subset=[strata_col, duration_col, event_type_col]).copy()
    sub = sub.loc[sub[duration_col] > 0]
    is_irae_event = (sub[event_type_col] == event_of_interest).astype(int)
    print(f"{title} (CIF):")
    kept, _dropped = _strata_min_events(
        sub.assign(_irae_event=is_irae_event), strata_col, duration_col, "_irae_event", min_events
    )
    if kept.empty:
        ax.text(0.5, 0.5, "(insufficient events after filtering)",
                ha="center", va="center", transform=ax.transAxes, color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    colors = dict(colors) if colors else {}
    for level, grp in kept.groupby(strata_col):
        ajf = AalenJohansenFitter()
        ajf.fit(grp[duration_col], grp[event_type_col], event_of_interest=event_of_interest,
                label=f"{level} (n={len(grp):,})")
        ajf.plot(ax=ax, color=colors.get(level))
    ax.set_title(title, fontsize=11, weight="bold")
    ax.set_xlabel("Days")
    ax.set_ylabel("Cumulative irAE incidence")
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7.5, loc="upper left")


## Figure 4a -- Overall time-to-irAE (unstratified) & by treatment type

Two standalone KM panels, landmark 0d:
1. **Overall** time-to-irAE curve across the whole cohort (no strata) -- the
   single-curve orientation slide before the stratified comparisons below.
2. **By treatment type** (drug class: PD1/PDL1 vs. CTLA4 vs. combination),
   same `drug_class_series` derivation used later in the small-multiples
   figure, shown here at full size as its own slide.

Both use the same `km_by_strata` helper (irAE-free probability, death
censored, log-rank p where applicable, `< MIN_EVENTS`-event arms dropped and
printed).


In [ ]:
KM_LANDMARK_HEADLINE = 0  # headline landmark; matches KM_LANDMARK defined below
cohort0_overall = COHORT.get(KM_LANDMARK_HEADLINE, pd.DataFrame()).copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# 1. Overall (unstratified) time-to-irAE.
if not cohort0_overall.empty:
    d = cohort0_overall.copy()
    d["_all"] = "All patients"
    km_by_strata(axes[0], d, "_all", title="Overall time-to-irAE")
else:
    axes[0].text(0.5, 0.5, "(no landmark-0 cohort data)", ha="center", va="center",
                 transform=axes[0].transAxes, color="#7f8c8d")
    axes[0].set_title("Overall time-to-irAE", fontsize=11, weight="bold")
    axes[0].set_axis_off()

# 2. By treatment type (drug class).
if not cohort0_overall.empty and {"pd1pdl1", "ctla4"}.issubset(cohort0_overall.columns):
    d = cohort0_overall.copy()
    d["_drug_class"] = drug_class_series(d)
    km_by_strata(axes[1], d, "_drug_class", title="Time-to-irAE by treatment type")
else:
    axes[1].text(0.5, 0.5, "(missing pd1pdl1/ctla4 columns)", ha="center", va="center",
                 transform=axes[1].transAxes, color="#7f8c8d")
    axes[1].set_title("Time-to-irAE by treatment type", fontsize=11, weight="bold")
    axes[1].set_axis_off()

fig.suptitle(f"Time-to-irAE -- landmark {KM_LANDMARK_HEADLINE}d", fontsize=13, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])

for ext in ("png",):
    out = OUT_DIR / f"km_overall_and_treatment_irae.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


In [ ]:
KM_LANDMARK = 0  # headline landmark; +90d panels live in the appendix cell below

cohort0 = COHORT.get(KM_LANDMARK, pd.DataFrame()).copy()

panel_specs = []  # list of (strata_col, title, prepared_df)

# 1. Drug class (pd1pdl1 vs ctla4 vs combination) -- descriptive only; combination
#    is not modeled (see Table 1 cell) but is shown here since KM is exploratory.
if not cohort0.empty and {"pd1pdl1", "ctla4"}.issubset(cohort0.columns):
    d = cohort0.copy()
    d["_drug_class"] = drug_class_series(d)
    panel_specs.append(("_drug_class", "Drug class", d))
else:
    print("KM panel 'Drug class': skipped, missing pd1pdl1/ctla4 columns")

# 2. Top univariate lab hits (median split) -- turns the volcano's top dots into curves.
if not cohort0.empty:
    top_labs, top_labs_detail = top_univariate_labs(KM_LANDMARK, k=2)
    print(f"Top univariate labs @ landmark {KM_LANDMARK}: {top_labs}")
    for lab in top_labs:
        if lab not in cohort0.columns:
            print(f"KM panel '{lab}': skipped, column not present in aggregated_landmark{KM_LANDMARK}.csv")
            continue
        strata_col = f"_lab_strata_{lab}"
        d = cohort0.copy()
        d[strata_col], cutoff = dichotomize(d, lab)
        panel_specs.append((strata_col, f"{lab} (median split @ {cutoff:.2g})", d))
else:
    print("KM panel 'top univariate labs': skipped, no landmark-0 cohort data")

# 3. Model risk tertiles -- test-set only, from the XGBoost patient-risks file.
risks0 = load_patient_risks(KM_LANDMARK, config="both")
if not cohort0.empty and not risks0.empty:
    merge_left = cohort0.reset_index() if cohort0.index.name else cohort0
    key = "DFCI_MRN" if "DFCI_MRN" in merge_left.columns else merge_left.columns[0]
    risk_key = "DFCI_MRN" if "DFCI_MRN" in risks0.columns else risks0.columns[0]
    merged = merge_left.merge(
        risks0[[risk_key, "risk_score"]], left_on=key, right_on=risk_key, how="inner"
    )
    if merged.empty:
        print("KM panel 'Model risk tertiles': skipped, no overlap between cohort and test-set risks")
    else:
        try:
            merged["_risk_tertile"] = pd.qcut(
                merged["risk_score"], 3, labels=["Low risk", "Mid risk", "High risk"]
            ).astype(str)
            panel_specs.append(("_risk_tertile", "Model risk tertiles (test set, XGBoost)", merged))
        except ValueError as exc:
            print(f"KM panel 'Model risk tertiles': skipped, could not bin into tertiles ({exc})")
else:
    print("KM panel 'Model risk tertiles': skipped, missing cohort or patient-risks data")

# 4. Genomic marker -- conditional on adequate group sizes; somatic indicators are
#    sparse, so this is skipped rather than forced when both arms can't clear MIN_EVENTS.
genomic0 = COHORT_GENOMIC.get(KM_LANDMARK, pd.DataFrame())
genomic_marker_cols = [c for c in genomic0.columns if GENOMIC_FEATURE_RE.match(str(c))] if not genomic0.empty else []
genomic_panel_added = False
for marker in genomic_marker_cols:
    d = genomic0.dropna(subset=[marker, "t_irae", "IRAE"]).copy()
    d[marker] = pd.to_numeric(d[marker], errors="coerce").fillna(0).astype(int)
    d["_marker_strata"] = d[marker].map({1: f"{marker} = 1", 0: f"{marker} = 0"})
    counts = d.groupby("_marker_strata")["IRAE"].sum()
    if (counts >= MIN_EVENTS).sum() >= 2:
        panel_specs.append(("_marker_strata", f"Genomic marker: {marker}", d))
        genomic_panel_added = True
        break  # one genomic panel is enough for a talk slide
if not genomic_panel_added:
    print(f"KM panel 'genomic marker': skipped, no somatic indicator has "
          f">= {MIN_EVENTS} events in >= 2 arms (checked {len(genomic_marker_cols)} markers)")

if panel_specs:
    fig, axes = plt.subplots(1, len(panel_specs), figsize=(6.5 * len(panel_specs), 5.5))
    if len(panel_specs) == 1:
        axes = [axes]
    for ax, (strata_col, title, d) in zip(axes, panel_specs):
        km_by_strata(ax, d, strata_col, title=title)
    fig.suptitle(f"Time-to-irAE, KM by univariate strata -- landmark {KM_LANDMARK}d", fontsize=13, weight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    for ext in ("png",):
        out = OUT_DIR / f"km_strata_irae.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
    plt.show()
else:
    print("KM small-multiples: no panels available to plot")


## Appendix -- CIF (competing-risk) backup + +90d KM robustness check

Backup slide for a survival-savvy audience: cumulative incidence of irAE
with death as a competing event (Aalen-Johansen), for the drug-class
stratum. Also re-runs the KM small-multiples at landmark +90d as a
robustness check.


In [ ]:
# --- CIF appendix: drug class, landmark 0d ---
cif_specs = [spec for spec in panel_specs if spec[0] == "_drug_class"]

if cif_specs:
    fig, axes = plt.subplots(1, len(cif_specs), figsize=(6.5 * len(cif_specs), 5.5))
    if len(cif_specs) == 1:
        axes = [axes]
    for ax, (strata_col, title, d) in zip(axes, cif_specs):
        cif_by_strata(ax, d, strata_col, title=f"{title} (CIF)")
    fig.suptitle("Cumulative irAE incidence, death as competing event -- landmark 0d",
                 fontsize=13, weight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    for ext in ("png",):
        out = OUT_DIR / f"cif_strata_irae.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
    plt.show()
else:
    print("CIF appendix: no drug-class panel available (see KM small-multiples cell above)")


# --- +90d KM robustness check ---
KM_LANDMARK_90 = 90
cohort90 = COHORT.get(KM_LANDMARK_90, pd.DataFrame()).copy()
panel_specs_90 = []

if not cohort90.empty and {"pd1pdl1", "ctla4"}.issubset(cohort90.columns):
    d = cohort90.copy()
    d["_drug_class"] = drug_class_series(d)
    panel_specs_90.append(("_drug_class", "Drug class", d))

if not cohort90.empty:
    top_labs_90, _ = top_univariate_labs(KM_LANDMARK_90, k=3)
    print(f"Top univariate labs @ landmark {KM_LANDMARK_90}: {top_labs_90}")
    for lab in top_labs_90:
        if lab not in cohort90.columns:
            continue
        strata_col = f"_lab_tertile_{lab}"
        d = cohort90.copy()
        d[strata_col], _ = tertile_split(d, lab)
        panel_specs_90.append((strata_col, f"{lab} (tertiles)", d))

risks90 = load_patient_risks(KM_LANDMARK_90, config="both")
if not cohort90.empty and not risks90.empty:
    merge_left = cohort90.reset_index() if cohort90.index.name else cohort90
    key = "DFCI_MRN" if "DFCI_MRN" in merge_left.columns else merge_left.columns[0]
    risk_key = "DFCI_MRN" if "DFCI_MRN" in risks90.columns else risks90.columns[0]
    merged90 = merge_left.merge(risks90[[risk_key, "risk_score"]], left_on=key, right_on=risk_key, how="inner")
    if not merged90.empty:
        try:
            merged90["_risk_tertile"] = pd.qcut(
                merged90["risk_score"], 3, labels=["Low risk", "Mid risk", "High risk"]
            ).astype(str)
            panel_specs_90.append(("_risk_tertile", "Model risk tertiles (test set, XGBoost)", merged90))
        except ValueError as exc:
            print(f"+90d KM panel 'Model risk tertiles': skipped ({exc})")

if panel_specs_90:
    fig, axes = plt.subplots(1, len(panel_specs_90), figsize=(6.5 * len(panel_specs_90), 5.5))
    if len(panel_specs_90) == 1:
        axes = [axes]
    for ax, (strata_col, title, d) in zip(axes, panel_specs_90):
        km_by_strata(ax, d, strata_col, title=title)
    fig.suptitle("Time-to-irAE, KM by univariate strata -- landmark +90d (robustness)",
                 fontsize=13, weight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    for ext in ("png",):
        out = OUT_DIR / f"km_strata_irae_90d.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
    plt.show()
else:
    print("+90d KM robustness check: no panels available to plot")


## Figure 2 -- Discrimination (Mean AUC(t))

The first output keeps the lab-arm comparison against clinical baseline at 0d
and +90d. The second output reads genomic-arm multivariate runs and compares
**labs-only**, **genomics-only**, and **genomics+labs** at 0d and +90d
(`GENOMIC_MULTIVARIATE_LANDMARKS = [0, 90]`). The genomics volcano/association
panel in Figure 1 stays 0d-only (`GENOMIC_ASSOCIATION_LANDMARKS = [0]`).

Sources (irae endpoint):
- Lab Cox/XGBoost: `cox|xgboost/landmark_{lm}/{both,baseline}/...metrics.csv`
- Genomic Cox/XGBoost: `genomic/cox|xgboost/landmark_{lm}/{labs,genomics,genomics_plus_labs}/...metrics.csv`

The genomic loader also accepts older/fallback output names such as `genomics/`,
`genomic_only/`, `genomic_plus_labs/`, and `both/`.

**Talk note:** the labs-vs-baseline panel (`discrimination_vs_baseline_irae`) is
talk-worthy on its own. The genomics comparison panel
(`discrimination_genomics_vs_genomics_labs_irae`, second plot below) is
**appendix if genomic signal weak** -- only promote it to the main deck if the
genomics/genomics+labs bars clearly beat labs-only.


In [ ]:
def _read_metrics(path, auc_col):
    df = pd.read_csv(path)
    row = df.loc[df["endpoint"] == "irae"]
    if row.empty:
        return np.nan
    row = row.iloc[0]
    return float(row[auc_col])


def _read_first_metrics(candidates, auc_col, *, label):
    for p in candidates:
        if p.exists():
            print(f"{label}: using {p}")
            return _read_metrics(p, auc_col)
    preview = "\n  ".join(str(p) for p in candidates[:12])
    if len(candidates) > 12:
        preview += f"\n  ... ({len(candidates) - 12} more)"
    print(f"{label}: missing; checked:\n  {preview}")
    return np.nan


def _metric_candidates(base_dirs, model_dir, landmark, config_dirs, filename):
    return [
        base_dir / model_dir / f"landmark_{landmark}" / config_dir / filename
        for base_dir in base_dirs
        for config_dir in config_dirs
    ]


def cox_metric_loader(base_dirs, config_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "cox", lm, config_dirs, "cox_agg_multivariable_metrics.csv"),
            "test_mean_auc_t",
            label=f"{label} Cox landmark {lm}",
        )
    return _load


def cox_baseline_loader(base_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "cox", lm, ["baseline"], "cox_agg_baseline_metrics.csv"),
            "test_mean_auc_t",
            label=f"{label} Cox baseline landmark {lm}",
        )
    return _load


def xgb_metric_loader(base_dirs, config_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "xgboost", lm, config_dirs, "landmark_xgboost_metrics.csv"),
            "mean_auc_t",
            label=f"{label} XGBoost landmark {lm}",
        )
    return _load


def xgb_baseline_loader(base_dirs, *, label):
    def _load(lm):
        return _read_first_metrics(
            _metric_candidates(base_dirs, "xgboost", lm, ["baseline"], "landmark_xgboost_baseline_metrics.csv"),
            "mean_auc_t",
            label=f"{label} XGBoost baseline landmark {lm}",
        )
    return _load


def plot_metric_comparison(series, landmarks, *, title, outfile_stem, figsize=(7, 5.5)):
    data = {name: [loader(lm) for lm in landmarks] for name, loader, _, _ in series}
    n_series = len(series)
    bar_width = min(0.20, 0.78 / max(n_series, 1))
    x_positions = np.arange(len(landmarks))
    offsets = (np.arange(n_series) - (n_series - 1) / 2) * bar_width

    fig, ax = plt.subplots(figsize=figsize)
    for i, (name, _loader, color, hatch) in enumerate(series):
        vals = [data[name][k] for k in range(len(landmarks))]
        bar_x = x_positions + offsets[i]
        ax.bar(bar_x, [v if np.isfinite(v) else 0.0 for v in vals], bar_width,
               color=color, hatch=hatch, edgecolor="white", linewidth=0.8, label=name)
        for x, v in zip(bar_x, vals):
            if np.isfinite(v):
                ax.text(x, v + 0.006, f"{v:.3f}", ha="center", va="bottom",
                        fontsize=6.5, weight="semibold", color=color, rotation=90)
    finite = [data[n][k] for n in data for k in range(len(landmarks)) if np.isfinite(data[n][k])]
    lo = min(0.45, (min(finite) if finite else 0.45) - 0.05)
    hi = min(1.05, (max(finite) if finite else 0.7) * 1.18)
    ax.set_ylim(lo, hi)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days" for lm in landmarks])
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
    ax.set_ylabel("Test Mean AUC(t)")
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="upper right", fontsize=7.5, ncol=2)

    fig.suptitle(title, fontsize=13, weight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    for ext in ("png",):
        out = OUT_DIR / f"{outfile_stem}.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
    plt.show()


LAB_SERIES = [
    ("Cox (labs)",          cox_metric_loader([BASE], ["both"], label="lab"),     "#4C72B0", None),
    ("Cox (baseline)",      cox_baseline_loader([BASE], label="lab"),              "#9DB3D6", "//"),
    ("XGBoost (labs)",      xgb_metric_loader([BASE], ["both"], label="lab"),     "#DD8452", None),
    ("XGBoost (baseline)",  xgb_baseline_loader([BASE], label="lab"),              "#EFC2A6", "//"),
]
plot_metric_comparison(
    LAB_SERIES,
    LANDMARKS,
    title="Cox vs. XGBoost, labs vs. baseline covariates - irae · landmark cohort",
    outfile_stem="discrimination_vs_baseline_irae",
)


GENOMIC_LABS_ONLY_CONFIGS = ["labs", "labs_only"]
GENOMIC_ONLY_CONFIGS = ["genomics", "genomic", "genomics_only", "genomic_only"]
GENOMIC_PLUS_LABS_CONFIGS = [
    "genomics_plus_labs",
    "genomic_plus_labs",
    "genomics_labs",
    "labs_plus_genomics",
    "both",
]
GENOMIC_SERIES = [
    ("Cox (labs)",                 cox_metric_loader(GENOMIC_BASES, GENOMIC_LABS_ONLY_CONFIGS, label="genomic"), "#9DB3D6", "\\"),
    ("Cox (genomics)",             cox_metric_loader(GENOMIC_BASES, GENOMIC_ONLY_CONFIGS, label="genomic"),      "#2471A3", "//"),
    ("Cox (genomics+labs)",        cox_metric_loader(GENOMIC_BASES, GENOMIC_PLUS_LABS_CONFIGS, label="genomic"), "#4C72B0", None),
    ("XGBoost (labs)",             xgb_metric_loader(GENOMIC_BASES, GENOMIC_LABS_ONLY_CONFIGS, label="genomic"), "#EFC2A6", "\\"),
    ("XGBoost (genomics)",         xgb_metric_loader(GENOMIC_BASES, GENOMIC_ONLY_CONFIGS, label="genomic"),      "#7D3C98", "//"),
    ("XGBoost (genomics+labs)",    xgb_metric_loader(GENOMIC_BASES, GENOMIC_PLUS_LABS_CONFIGS, label="genomic"), "#DD8452", None),
]
plot_metric_comparison(
    GENOMIC_SERIES,
    GENOMIC_MULTIVARIATE_LANDMARKS,
    title="Labs vs. genomics vs. genomics+labs - irae · genomic cohort",
    outfile_stem="discrimination_genomics_vs_genomics_labs_irae",
    figsize=(7, 5.5),
)


## Figure 3 -- Model importance (Cox + XGBoost)

Two separate figures, one per landmark (0d and +90d). Each figure has two
panels: Cox log-HR coefficients (left) and XGBoost gain (right). Baseline
clinical covariates (GENDER/AGE/pd1pdl1/ctla4/CANCER_TYPE_*) are excluded
from both panels so they focus on lab-derived features.

Sources (irae endpoint):
- **Cox** `cox/landmark_{lm}/both/cox_agg_multivariable.csv` -> column `coef`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_feature_importance.csv` -> column `gain`

Saved as `model_importance_irae_landmark0.{png,pdf}` and
`model_importance_irae_landmark90.{png,pdf}`.

**Talk note:** show the 0d figure (Elastic-Net Cox panel especially -- the
single most interpretable importance view) in the talk deck; the +90d
figure is **appendix**.


In [ ]:
BASELINE_COVARIATE_NAMES = {"age", "GENDER_MALE", "pd1pdl1", "ctla4"}


def _is_baseline_covariate(feature: str) -> bool:
    return feature in BASELINE_COVARIATE_NAMES or str(feature).startswith("CANCER_TYPE_")


def load_cox_coefs(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    # Exclude baseline clinical covariates (GENDER/AGE/pd1pdl1/ctla4/CANCER_TYPE_*)
    # so the panel focuses on lab-derived features. The exact flag column name
    # depends on multivariate_analysis.py's output schema -- check a couple of
    # likely names defensively, falling back to the explicit name/prefix check.
    baseline_flag_col = next(
        (c for c in ("is_baseline_covariate", "is_age_covariate", "is_clinical_covariate")
         if c in df.columns),
        None,
    )
    if baseline_flag_col is not None:
        df = df.loc[~df[baseline_flag_col].fillna(False).astype(bool)]
    if "feature" in df.columns:
        df = df.loc[~df["feature"].map(_is_baseline_covariate)]
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def load_xgb_importance(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_feature_importance.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "irae"].copy()
    df = df.loc[~df["feature"].map(_is_baseline_covariate)]
    df = df.loc[df["gain"].fillna(0) > 0]
    return df


def render_importance_panel(ax, df, title, *, value_col, xlabel, signed):
    if df.empty:
        ax.text(0.5, 0.5, "(no features to display)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    df = df.copy()
    if "lab_name" not in df.columns:
        parsed = df["feature"].map(parse_feature)
        df["lab_name"] = parsed.str[0]
        df["feature_stat"] = parsed.str[1]
    df["category"] = df["lab_name"].map(assign_category) if "lab_name" in df.columns else "Other"

    sort_key = df[value_col].abs() if signed else df[value_col]
    df = df.reindex(sort_key.sort_values(ascending=False).index).head(TOP_N)
    df = df.iloc[::-1]
    values = df[value_col].to_numpy()

    colors = [CATEGORY_COLORS.get(c, CATEGORY_COLORS["Other"]) for c in df["category"]]
    y = np.arange(len(df))
    ax.barh(y, values, color=colors, edgecolor="white", linewidth=0.6)
    if signed:
        ax.axvline(0, color="black", linewidth=0.5, zorder=0)

    if "feature_stat" in df.columns:
        labels = [format_label(r["lab_name"], r["feature_stat"]) for _, r in df.iterrows()]
    elif "lab_name" in df.columns:
        labels = df["lab_name"].tolist()
    else:
        labels = df["feature"].tolist() if "feature" in df.columns else [str(i) for i in df.index]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, weight="bold")


def plot_importance_landmark(lm):
    """One figure per landmark: Cox (left) + XGBoost (right) importance panels."""
    sign = "+" if lm > 0 else ""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)

    ax = axes[0]
    title = f"Elastic-Net Cox  ·  {sign}{lm} days"
    try:
        df = load_cox_coefs(lm)
        render_importance_panel(ax, df, title, value_col="coef", xlabel="log HR coefficient", signed=True)
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=8, color="#c0392b")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()

    ax = axes[1]
    title = f"XGBoost  ·  {sign}{lm} days"
    try:
        df = load_xgb_importance(lm)
        render_importance_panel(ax, df, title, value_col="gain", xlabel="gain", signed=False)
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=8, color="#c0392b")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()

    handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
               for c in LEGEND_ORDER]
    fig.legend(handles=handles, loc="lower center",
               ncol=len(handles), bbox_to_anchor=(0.5, -0.06),
               fontsize=10)

    stem = f"model_importance_irae_landmark{lm}"
    for ext in ("png",):
        out = OUT_DIR / f"{stem}.{ext}"
        fig.savefig(out)
        print(f"wrote {out}")
    plt.show()


for lm in LANDMARKS:
    plot_importance_landmark(lm)


## Figure 5 -- PGS interpretation (out of scope)

COMPASS's Figure 4 plots PGS-only Cox HR forest plots for Testosterone and
PSA polygenic scores. IPIO has no PGS data for this cohort, so that figure is
omitted entirely here.
